## Data Generation: IoU Scoring and Interpolation Sweeps

This notebook generates the core CSV artifacts used by downstream plotting and replacement experiments.

### Part 1
Compute IoU scores for 71 programs (70 golden programs + uniform lower diagonal) across BERT, GPT-2, and TinyLlama.

### Part 2
Run linear interpolation sweeps with k = 1..20 top-ranked programs per head.

### Output Files
- ../data/iou_scores_bert.csv
- ../data/iou_scores_gpt2.csv
- ../data/iou_scores_tinyllama.csv
- ../data/interpolation_k_bert.csv
- ../data/interpolation_k_gpt2.csv
- ../data/interpolation_k_tinyllama.csv

In [ ]:
# Install dependencies if needed
# !pip install torch transformers spacy nltk tqdm scikit-learn
# !python -m spacy download en_core_web_sm


In [8]:
import os
import csv
import json
import time
import traceback
import warnings
import inspect
import random
from pathlib import Path
from typing import Callable, Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import torch
import spacy
from tqdm import tqdm
from transformers import (
    AutoTokenizer, AutoModel,
    PreTrainedModel, PreTrainedTokenizerBase
)
from sklearn.linear_model import LinearRegression

warnings.filterwarnings("ignore")

# Reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

# Project-relative paths (notebook lives in code/)
DATA_DIR = "../data"
RESULTS_DIR = "../results"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

nlp = spacy.load("en_core_web_sm")
print(f"[INFO] Imports ready. DATA_DIR={DATA_DIR} | RESULTS_DIR={RESULTS_DIR}")

[INFO] Imports ready. DATA_DIR=../data | RESULTS_DIR=../results


In [9]:
# ── Environment flag ─────────────────────────────────────────────────────────
# Set RUNNING_ON to "local" or "colab" before running the notebook.
#   "local"  → uses relative ../data paths, CPU-only models
#   "colab"  → mounts Google Drive, uses GPU, can load gated/large models
#
# Colab checklist:
#   1. Runtime → Change runtime type → GPU (T4 or better)
#   2. Paste your HuggingFace token in HF_TOKEN if accessing gated models
#   3. Confirm DRIVE_ROOT points to the right folder in your Drive

RUNNING_ON = "local"   # ← change to "colab" when on Colab

# ── HuggingFace auth (needed for gated models, e.g. Llama) ───────────────────
HF_TOKEN: str | None = None   # e.g. "hf_xxxxxxxxxxxxxxxxxxxx"

if RUNNING_ON == "colab":
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT  = "/content/drive/MyDrive/attention_interp"  # ← adjust if needed
    DATA_DIR    = f"{DRIVE_ROOT}/data"
    RESULTS_DIR = f"{DRIVE_ROOT}/results"
else:
    DATA_DIR    = "../data"
    RESULTS_DIR = "../results"

import os
os.makedirs(DATA_DIR,    exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# Device selection
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[INFO] RUNNING_ON={RUNNING_ON!r} | DEVICE={DEVICE}")
print(f"[INFO] DATA_DIR={DATA_DIR} | RESULTS_DIR={RESULTS_DIR}")


[INFO] RUNNING_ON='local' | DEVICE=cpu
[INFO] DATA_DIR=../data | RESULTS_DIR=../results


In [10]:
# ── Scoring helpers ──────────────────────────────────────────────────────────

def iou_score(p: np.ndarray, q: np.ndarray) -> float:
    """IoU similarity between two attention distributions (higher = better)."""
    p = np.clip(p, 1e-12, 1.0)
    q = np.clip(q, 1e-12, 1.0)
    intersection = np.sum(np.minimum(p, q))
    union        = np.sum(np.maximum(p, q))
    return float(intersection / union) if union > 0 else 0.0


def score_head_iou(
    model: PreTrainedModel,
    tokenizer: PreTrainedTokenizerBase,
    layer: int,
    head: int,
    pattern: Callable,
    sentence: str,
) -> float:
    """
    Returns mean row-wise IoU between actual attention and predicted pattern.
    Returns NaN on any error so callers can handle gracefully.
    """
    try:
        tokens = tokenizer(sentence, return_tensors="pt")
        with torch.no_grad():
            att = model(**tokens, output_attentions=True).attentions[layer][0, head].detach().numpy()

        # detect paradigm: 1-arg (Jacob's) vs 2-arg (sentence, tokenizer)
        n_params = len(inspect.signature(pattern).parameters)
        if n_params == 1:
            _, pred_att = pattern(sentence)
        else:
            _, pred_att = pattern(sentence, tokenizer)

        pred_att = np.array(pred_att, dtype=float)
        min_len = min(att.shape[0], pred_att.shape[0])
        att      = att[:min_len, :min_len]
        pred_att = pred_att[:min_len, :min_len]

        row_ious = [iou_score(att[i], pred_att[i]) for i in range(min_len)]
        return float(np.mean(row_ious))

    except Exception:
        return float("nan")


In [33]:
import importlib
import data.golden_programs_two

# After you've modified the file:
importlib.reload(data.golden_programs_two)

# Now re-run your inspection logic
import inspect
from data import golden_programs_two

# Get all functions from the module
all_programs = [
    obj for _, obj in inspect.getmembers(golden_programs_two, inspect.isfunction)
]
remove = [
    "_get_gpt2_tokenizer", "align_gpt2_to_spacy", "align_spacy_to_gpt2", 
    "apply_causal_mask", "spacy_parse", "first_token_bias_stochastic_L7H11", 
    "get_modifying_adjectives", "get_nlp", "gpt2_tokenize", "make_row_stochastic",]
all_programs = [f for f in all_programs if f.__name__ not in remove]

In [23]:
# --- Load golden programs + uniform_lower_diagonal ---
import sys
import os
import inspect
import json
from typing import List, Callable
module_root = os.path.abspath("..")
if module_root not in sys.path:
    sys.path.append(module_root)

from data import golden_programs_two

all_programs: List[Callable] = [
    obj for _, obj in inspect.getmembers(golden_programs_two, inspect.isfunction)
]

# Save index mapping so CSV program_idx values are always interpretable
PROGRAM_INDEX_MAP = {i: p.__name__ for i, p in enumerate(all_programs)}
print(f"[INFO] Total programs loaded: {len(all_programs)}")

for idx, name in PROGRAM_INDEX_MAP.items():
    print(f"  {idx}: {name}")
# print(f"[INFO] Saving program index map to {DATA_DIR}/program_index_map.json")

# with open(f"{DATA_DIR}/program_index_map.json", "w", encoding="utf-8") as _f:
#     json.dump(PROGRAM_INDEX_MAP, _f, indent=2)

print("[DONE] Program index map written.")

[INFO] Total programs loaded: 144
  0: adverbial_modulation
  1: appositive_phrase_attention
  2: cls_attention
  3: complement_adjunct_relationship
  4: compound_element_association
  5: compound_modifier_attention
  6: compound_word_attention_pattern
  7: conjunction_based_grouping
  8: conjunction_resolution
  9: contextual_anchoring
  10: coord_and_verb_dependency
  11: coordination_attention
  12: coreference_resolution
  13: decaying_content_focus_L6H1
  14: decaying_content_focus_punctuation_coreference_L0H1
  15: decaying_first_token_bias_L10H11
  16: decaying_first_token_bias_L11H9
  17: decaying_first_token_bias_L1H3
  18: decaying_first_token_bias_content_focus_L0H0
  19: decaying_first_token_bias_content_focus_L0H10
  20: decaying_first_token_bias_content_focus_L11H3
  21: decaying_first_token_bias_content_focus_L4H0
  22: decaying_first_token_bias_content_focus_L4H8
  23: decaying_first_token_bias_content_focus_L4H9
  24: decaying_first_token_bias_content_focus_L7H1
  25: 

In [17]:
# --- Load generic sentences and sample 5 (seeded random sample) ---
df_json = pd.read_json(f"{DATA_DIR}/generic_sentences.json")
generic_sentences_all = df_json[0].tolist()

_rng = random.Random(RANDOM_SEED)
_shuffled = generic_sentences_all[:]
_rng.shuffle(_shuffled)
SAMPLE_SENTENCES: List[str] = _shuffled[:5]

# Save mapping so sentence_idx values in CSVs remain recoverable
SENTENCE_INDEX_MAP = {i: s for i, s in enumerate(SAMPLE_SENTENCES)}
print(f"[INFO] Total candidate sentences: {len(generic_sentences_all)}")
print(f"[INFO] Sample size: {len(SAMPLE_SENTENCES)}")
for idx, sentence in SENTENCE_INDEX_MAP.items():
    print(f"  {idx}: {sentence}")
print(f"[INFO] Saving sentence index map to {DATA_DIR}/sentence_index_map.json")

with open(f"{DATA_DIR}/sentence_index_map.json", "w", encoding="utf-8") as _f:
    json.dump(SENTENCE_INDEX_MAP, _f, indent=2)

print("[DONE] Sentence index map written.")

[INFO] Total candidate sentences: 100
[INFO] Sample size: 5
  0: The rhythmic crash of the waves, breaking against the shore, provided a soothing soundtrack to the quiet evening.
  1: He contemplated, for a moment, the vastness of the universe, its endless possibilities, and his tiny place within it.
  2: 'That's an excellent point!' she agreed, nodding enthusiastically, acknowledging the validity of his argument.
  3: If you truly want to succeed, you must work diligently, persevere through challenges, and never give up on your dreams.
  4: He meticulously organized his tools: wrenches, screwdrivers, pliers, and a tape measure, all in their designated spots.
[INFO] Saving sentence index map to ../data/sentence_index_map.json
[DONE] Sentence index map written.


In [41]:
# ── Model configurations ─────────────────────────────────────────────────────
# Each entry supports optional fields:
#   use_fp16       – load in float16 (saves ~50% VRAM, needed for 3B+ on Colab)
#   use_device_map – offload layers across CPU/GPU automatically
#   hf_token       – override global HF_TOKEN per model
#   attn_impl      – "eager" forces attention weight collection (required for
#                    flash-attention-based models; GQA handled transparently)

_LOCAL_MODELS = {
    "bert": {
        "model_name": "bert-base-uncased",
        "out_csv":    f"{DATA_DIR}/iou_scores_bert.csv",
    },
    # "gpt2": {
    #     "model_name": "openai-community/gpt2",
    #     "out_csv":    f"{DATA_DIR}/iou_scores_gpt2.csv",
    # },
    "tinyllama": {
        "model_name": "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T",
        "out_csv":    f"{DATA_DIR}/iou_scores_tinyllama.csv",
        "attn_impl":  "eager",
        "use_fp16":   False,
    },
}

_COLAB_MODELS = {
    **_LOCAL_MODELS,
    "llama3b": {
        "model_name":   "meta-llama/Llama-3.2-3B",
        "out_csv":      f"{DATA_DIR}/iou_scores_llama3b.csv",
        "attn_impl":    "eager",   # flash-attn does not expose attention weights
        "use_fp16":     True,      # ~6 GB VRAM in fp16 vs ~12 GB in fp32
        "use_device_map": True,
    },
}

MODEL_CONFIGS = _COLAB_MODELS if RUNNING_ON == "colab" else _LOCAL_MODELS

print("[INFO] Model configurations loaded.")
for k, cfg in MODEL_CONFIGS.items():
    print(f"  {k:<12}: {cfg['model_name']}")


[INFO] Model configurations loaded.
  bert        : bert-base-uncased
  tinyllama   : TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T


In [39]:
# ── Category definitions & centroid functions ────────────────────────────────

from typing import Dict, List, Tuple, Callable
import numpy as np
from transformers import PreTrainedTokenizerBase

# FINAL CATEGORIZATION FOR 71 PROGRAMS
PROGRAM_CATEGORIES: Dict[str, List[str]] = {
    "strict_anchors": [
        "initial_token_anchoring", "cls_attention", "sentence_beginning_salience",
        "first_token_domination", "head_initial_token_emphasis",
        "initial_token_attachment", "initial_token_reference_attention", "initial_token_emphasis",
        "initial_token_centralization", "leading_contextual_emphasis", "sentence_start_dominance",
        "initial_phrase_contextualization", "initial_element_reinforcement", "token_reinforcement",
        "sentence_initiation_emphasis", "initial_token_dominance", "sentence_boundary_focus",
        "sentence_level_attention", "token_emphasis_subsequent_dominance", 
        "dominant_subject_attention", "initial_contextual_attention", 
        "initial_phrase_dominance", "semantic_grouping",
        "sentence_beginning_attention_pattern", "sentence_beginning_emphasis",
        "sentence_level_initial_token_repetition", "sentence_opening_salience",
        "initial_reference_attention", 
        "decaying_content_focus_L6H1", "decaying_first_token_bias_L11H9", "decaying_first_token_bias_L10H11", "decaying_first_token_bias_content_focus_L4H8", "decaying_first_token_bias_content_focus_L7H1", "decaying_first_token_bias_content_focus_punctuation_L11H1", "decaying_first_token_bias_content_focus_punctuation_L11H10", "decaying_first_token_bias_content_focus_punctuation_L5H11", "decaying_first_token_bias_content_focus_punctuation_L9H2", "decaying_first_token_bias_content_focus_punctuation_stochastic_L11H7", "decaying_first_token_bias_punctuation_L2H6", "decaying_first_token_bias_punctuation_L5H8", "decaying_first_token_bias_punctuation_L7H7", "decaying_first_token_bias_stochastic_L6H6", "first_token_bias_L10H2", "first_token_bias_L10H4", "first_token_bias_L10H6", "first_token_bias_L11H6", "first_token_bias_L3H4", "first_token_bias_L5H1", "first_token_bias_L6H9", "first_token_bias_L7H10", "first_token_bias_L7H2", "first_token_bias_L8H1", "first_token_bias_L9H1", "first_token_bias_L9H4", "first_token_bias_L9H5", "first_token_bias_L9H6", "first_token_bias_content_focus_L10H10", "first_token_bias_content_focus_L3H5", "first_token_bias_content_focus_L5H5", "first_token_bias_content_focus_L5H9", "first_token_bias_content_focus_L7H5", "first_token_bias_content_focus_L8H3", "first_token_bias_content_focus_coreference_L4H4", "first_token_bias_content_focus_punctuation_L10H0", "first_token_bias_content_focus_punctuation_L10H1", "first_token_bias_content_focus_punctuation_L11H2", "first_token_bias_content_focus_punctuation_L11H5", "first_token_bias_content_focus_punctuation_L5H0", "first_token_bias_content_focus_punctuation_L8H6", "first_token_bias_content_focus_punctuation_coreference_L4H2", "first_token_bias_content_focus_punctuation_stochastic_L3H1", "first_token_bias_content_focus_stochastic_L5H2", "first_token_bias_content_focus_stochastic_L6H5", "first_token_bias_punctuation_L10H8", "first_token_bias_punctuation_L11H4", "first_token_bias_punctuation_L1H8", "first_token_bias_punctuation_L4H10", "first_token_bias_punctuation_L6H10", "first_token_bias_punctuation_L8H11", "first_token_bias_punctuation_L9H11", "first_token_bias_punctuation_stochastic_L1H6", "first_token_bias_stochastic_L3H0", "first_token_bias_stochastic_L9H9"
    ],
    "broadcasts_and_global": [
        "adverbial_modulation", "appositive_phrase_attention",
        "compound_modifier_attention", "compound_word_attention_pattern",
        "high_saliency_relationship_detection", "parenthetical_attention",
        "parenthetical_insertion", "quotation_association",
        "uniform_attention", "relative_position_attention",
    ],
    "structural_and_boundaries": [
        "eos_attention", "last_token_attention", "special_token_attention",
        "sentence_initial_dominance", "punctuation_attention",
        "complement_adjunct_relationship", "coord_and_verb_dependency",
        "coordination_attention", "coreference_resolution",
        "emphasize_verbs_and_objects", "pronoun_reference"
    ],
    "relative_flow": [
        "next_attention", "previous_attention", "dependencies"
    ],
    "identity_and_local": [
        "same_attention", "lexical_diversity_focus",
        "rare_word_dominance", "conjunction_resolution", "pos_alignment",
        "repeated_attention", "first_token_dominance"
    ],
    "global_uniform": [
        "uniform_lower_diagonal", "contextual_anchoring", "sentence_position_preference",
        "decaying_first_token_bias_content_focus_L0H0", "decaying_first_token_bias_content_focus_L11H3", "decaying_first_token_bias_content_focus_L4H0", "decaying_first_token_bias_content_focus_L4H9", "decaying_first_token_bias_content_focus_L7H3", "decaying_first_token_bias_content_focus_punctuation_L0H11", "decaying_first_token_bias_content_focus_punctuation_L2H5", "decaying_stochastic_L1H10", "first_token_bias_L9H10", "first_token_bias_content_focus_L10H5", "first_token_bias_content_focus_L6H3", "first_token_bias_content_focus_L8H8", "first_token_bias_content_focus_punctuation_L0H7", "first_token_bias_content_focus_punctuation_L3H8", "first_token_bias_content_focus_punctuation_L6H2"
    ],
    "sentence_start": [
        "sentence_start_emphasis", "first_token_emphasis", "initial_token_attention",
        "main_subject_attention", "sentence_start_attention"
    ],
    "compound_element": [
        "compound_element_association", "negation_attention", "initial_word_attention",
        "conjunction_based_grouping"
    ]
}

# ── Centroid functions ────────────────────────────────────────────────────────

def _n(sentence, tokenizer):
    return len(tokenizer([sentence], return_tensors="pt").input_ids[0])

def _norm(out):
    return out / np.clip(out.sum(axis=1, keepdims=True), 1e-9, None)

def get_prog_fn(name: str):
    for fn in all_programs:
        if fn.__name__ == name:
            return fn
    raise NameError(f"Program function '{name}' not found in all_programs list.")

def strict_anchoring_centroid(sentence: str, tokenizer: PreTrainedTokenizerBase) -> Tuple[str, np.ndarray]:
    """VERTICAL Centroid: Pure focus on index 0."""
    n = len(tokenizer([sentence], return_tensors="pt").input_ids[0])
    out = np.zeros((n, n))
    out[:, 0] = 1.0 # The "CLS" or "Start" bar
    return "Strict Anchoring Centroid", out

def broadcasts_global_centroid(sentence: str, tokenizer: PreTrainedTokenizerBase) -> Tuple[str, np.ndarray]:
    """broadcasts_and_global: flat uniform – every token equally attended."""
    n = _n(sentence, tokenizer)
    return "Broadcasts/Global Centroid", np.ones((n, n)) / n


def eos_anchor_centroid(sentence: str, tokenizer: PreTrainedTokenizerBase) -> Tuple[str, np.ndarray]:
    """structural_and_boundaries: all tokens attend to the last position."""
    n = _n(sentence, tokenizer)
    out = np.zeros((n, n))
    out[:, -1] = 1.0
    return "EOS Anchor Centroid", _norm(out)


def relative_flow_centroid(sentence: str, tokenizer: PreTrainedTokenizerBase) -> Tuple[str, np.ndarray]:
    """relative_flow: symmetric ±1 off-diagonal (equal prev / next weight).
    Changed from asymmetric (0.6 prev, 0.4 next) which gave poor coherence."""
    n = _n(sentence, tokenizer)
    out = np.zeros((n, n))
    for i in range(n):
        if i > 0:     out[i, i - 1] = 0.5
        if i < n - 1: out[i, i + 1] = 0.5
    return "Relative Flow Centroid", _norm(out)


def identity_centroid(sentence: str, tokenizer: PreTrainedTokenizerBase) -> Tuple[str, np.ndarray]:
    """identity_and_local: pure self-attention diagonal."""
    n = _n(sentence, tokenizer)
    return "Identity Centroid", np.eye(n)


def dynamic_spike_centroid(sentence: str, tokenizer: PreTrainedTokenizerBase) -> Tuple[str, np.ndarray]:
    """
    Handles the 17 'Semantic/Syntactic Spike' programs.
    Instead of a fixed pattern, it targets the diagonal + noun/verb positions.
    """
    n = len(tokenizer([sentence], return_tensors="pt").input_ids[0])
    out = np.eye(n) * 0.5
    # Simulate a dynamic 'spike' behavior that these programs share
    if n > 2:
        out[:, 1] += 0.3  # Often the root/subject
        out[1, :] += 0.2
    out /= np.clip(out.sum(axis=1, keepdims=True), a_min=1e-9, a_max=None)
    return "Semantic/Syntactic Spike Centroid", out


def causal_uniform_centroid(sentence: str, tokenizer: PreTrainedTokenizerBase) -> Tuple[str, np.ndarray]:
    """global_uniform: lower-triangular uniform (causal mask archetype).
    Fixed from flat uniform which scored only 0.409 vs uniform_lower_diagonal."""
    n = _n(sentence, tokenizer)
    out = np.tril(np.ones((n, n)))
    return "Causal Uniform Centroid", _norm(out)

def sentence_start_centroid(s, t):
    return "Start Emphasis", get_prog_fn("sentence_start_emphasis")(s, t)[1]

def compound_element_centroid(s, t):
    return "Compound Element", get_prog_fn("compound_element_association")(s, t)[1]
# return "Dep Centroid", get_prog_fn("dependencies")(s, t)[1]

CATEGORY_CENTROIDS: Dict[str, Callable] = {
    "strict_anchors":              strict_anchoring_centroid,
    "broadcasts_and_global":       broadcasts_global_centroid,
    "structural_and_boundaries":   eos_anchor_centroid,
    "relative_flow":               relative_flow_centroid,
    "identity_and_local":          identity_centroid,
    # "semantic_and_syntactic_spikes": dynamic_spike_centroid,
    "global_uniform":              causal_uniform_centroid,
    "sentence_start":               sentence_start_centroid,
    "compound_element":             compound_element_centroid
}
assert set(CATEGORY_CENTROIDS) == set(PROGRAM_CATEGORIES), "Every category needs a centroid!"

# ── Reverse lookup & index map ────────────────────────────────────────────────
PROGRAM_TO_CATEGORY: Dict[str, str] = {
    prog: cat for cat, progs in PROGRAM_CATEGORIES.items() for prog in progs
}

_name_to_idx = {fn.__name__: i for i, fn in enumerate(all_programs)}
CATEGORY_PROG_INDICES: Dict[str, List[int]] = {}
for cat, prog_names in PROGRAM_CATEGORIES.items():
    indices = []
    for name in prog_names:
        if name in _name_to_idx:
            indices.append(_name_to_idx[name])
        else:
            print(f"[WARN] '{name}' in '{cat}' not found in all_programs.")
    CATEGORY_PROG_INDICES[cat] = sorted(indices)

categorized   = {i for idxs in CATEGORY_PROG_INDICES.values() for i in idxs}
uncategorized = [all_programs[i].__name__ for i in range(len(all_programs)) if i not in categorized]
print(f"[INFO] Programs categorized : {len(categorized)} / {len(all_programs)}")
if uncategorized:
    print(f"[WARN] Uncategorized: {uncategorized}")
for cat, idxs in CATEGORY_PROG_INDICES.items():
    print(f"  {cat:<30}: {len(idxs):>2} programs")


[WARN] 'first_token_bias_L9H4' in 'strict_anchors' not found in all_programs.
[WARN] 'uniform_lower_diagonal' in 'global_uniform' not found in all_programs.
[INFO] Programs categorized : 139 / 143
[WARN] Uncategorized: ['decaying_content_focus_punctuation_coreference_L0H1', 'decaying_first_token_bias_L1H3', 'decaying_first_token_bias_content_focus_L0H10', 'first_token_bias_punctuation_L9H4']
  strict_anchors                : 82 programs
  broadcasts_and_global         : 10 programs
  structural_and_boundaries     : 11 programs
  relative_flow                 :  3 programs
  identity_and_local            :  7 programs
  global_uniform                : 17 programs
  sentence_start                :  5 programs
  compound_element              :  4 programs


In [42]:
# ── Category exploration: validate centroids and check placement ──────────────
# Run every program on the first sample sentence, compute IoU vs each centroid,
# and flag any program whose best centroid differs from its assigned category.
# Requires only a lightweight tokenizer (BERT) – no full model forward pass.

_probe_sentence = SAMPLE_SENTENCES[0]
_probe_tok      = AutoTokenizer.from_pretrained("bert-base-uncased")

# Pre-compute program matrices on probe sentence
print(f"[INFO] Probe: {_probe_sentence!r}")
print(f"[INFO] Computing {len(all_programs)} program outputs...")
prog_matrices_probe: Dict[int, np.ndarray] = {}
for p_idx, prog in enumerate(all_programs):
    try:
        n_params = len(inspect.signature(prog).parameters)
        _, mat   = prog(_probe_sentence) if n_params == 1 else prog(_probe_sentence, _probe_tok)
        prog_matrices_probe[p_idx] = np.array(mat, dtype=float)
    except Exception as e:
        print(f"  [WARN] {prog.__name__}")
print(f"[INFO] {len(prog_matrices_probe)} / {len(all_programs)} program matrices computed.")

# Pre-compute centroid matrices on probe sentence
centroid_matrices_probe: Dict[str, np.ndarray] = {}
for cat, fn in CATEGORY_CENTROIDS.items():
    _, cmat = fn(_probe_sentence, _probe_tok)
    centroid_matrices_probe[cat] = np.array(cmat, dtype=float)

categories_ordered = list(CATEGORY_CENTROIDS.keys())

# Per-program IoU vs every centroid
header = f"{'':1} {'Program':<42}" + "".join(f"  {c[:9]:>9}" for c in categories_ordered)
header += f"  {'Assigned':>18}  {'Best centroid':>22}  {'Match':>5}"
print("\n" + header)
print("-" * len(header))

mismatches = []
for p_idx, prog in enumerate(all_programs):
    if p_idx not in prog_matrices_probe:
        continue
    mat          = prog_matrices_probe[p_idx]
    assigned_cat = PROGRAM_TO_CATEGORY.get(prog.__name__, "UNCATEGORIZED")
    scores: Dict[str, float] = {}
    for cat, cmat in centroid_matrices_probe.items():
        min_len = min(mat.shape[0], cmat.shape[0])
        row_ious = [iou_score(mat[i, :min_len], cmat[i, :min_len]) for i in range(min_len)]
        scores[cat] = float(np.mean(row_ious))
    best_cat = max(scores, key=scores.__getitem__)
    match    = "✓" if best_cat == assigned_cat else "⚠"
    if best_cat != assigned_cat:
        mismatches.append((prog.__name__, assigned_cat, best_cat))
    row = f"{match} {prog.__name__:<42}"
    row += "".join(f"  {scores[c]:>9.3f}" for c in categories_ordered)
    row += f"  {assigned_cat:>18}  {best_cat:>22}  {match:>5}"
    print(row)

print(f"\n[INFO] Mismatches (best centroid ≠ assigned category): {len(mismatches)}")
for name, assigned, best in mismatches:
    print(f"  {name}: assigned={assigned}, best centroid={best}")

# Within-category centroid IoU summary
print("\n[INFO] Within-category centroid coherence:")
print(f"  {'Category':<25}  {'N':>3}  {'mean':>6}  {'min':>6}  {'max':>6}")
for cat, idxs in CATEGORY_PROG_INDICES.items():
    cmat   = centroid_matrices_probe[cat]
    ious   = []
    for p_idx in idxs:
        if p_idx not in prog_matrices_probe:
            continue
        mat = prog_matrices_probe[p_idx]
        min_len = min(mat.shape[0], cmat.shape[0])
        ious.append(float(np.mean([iou_score(mat[i, :min_len], cmat[i, :min_len]) for i in range(min_len)])))
    if ious:
        print(f"  {cat:<25}  {len(ious):>3}  {np.mean(ious):>6.3f}  {np.min(ious):>6.3f}  {np.max(ious):>6.3f}")
    else:
        print(f"  {cat:<25}  ---")


[INFO] Probe: 'The rhythmic crash of the waves, breaking against the shore, provided a soothing soundtrack to the quiet evening.'
[INFO] Computing 143 program outputs...
[INFO] 143 / 143 program matrices computed.

  Program                                     strict_an  broadcast  structura  relative_  identity_  global_un  sentence_  compound_            Assigned           Best centroid  Match
---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
✓ adverbial_modulation                            0.107      0.915      0.020      0.042      0.064      0.408      0.045      0.087  broadcasts_and_global   broadcasts_and_global      ✓
✓ appositive_phrase_attention                     0.107      0.915      0.020      0.042      0.064      0.408      0.045      0.087  broadcasts_and_global   broadcasts_and_global      ✓
✓ cls_attention                            

In [43]:
# ── PART 1: Category-aware IoU scoring ───────────────────────────────────────
#
# Design:
#   Step 1 – 1 forward pass per sentence → cache all layer×head attention tensors
#   Step 2 – cache all 71 program matrices (pure numpy, no model)
#   Step 3 – score each (layer, head) vs every centroid → assign best category
#   Step 4 – score only in-category programs (≈7–14 instead of 71)
#
# GQA note (Llama 3B):
#   HuggingFace expands KV heads to match Q heads before returning attention
#   weights, so outputs.attentions[l] is always [batch, num_q_heads, S, S].
#   No special-casing needed; we do add .float() and .cpu() for safety.
#
# CSV schema: layer, head, program_idx, sentence_idx, iou_score  (unchanged)

from collections import Counter

PART1_COLUMNS = ["layer", "head", "program_idx", "sentence_idx", "iou_score"]


def _load_model(cfg: dict):
    """Load model with per-config options; returns (model, tokenizer)."""
    from transformers import AutoModel, AutoTokenizer
    kwargs = dict(output_attentions=True)
    if cfg.get("attn_impl"):
        kwargs["attn_implementation"] = cfg["attn_impl"]
    if cfg.get("use_fp16"):
        kwargs["torch_dtype"] = torch.float16
    if cfg.get("use_device_map"):
        kwargs["device_map"] = "auto"
    tok_kwargs = {}
    token = cfg.get("hf_token") or HF_TOKEN
    if token:
        kwargs["token"]     = token
        tok_kwargs["token"] = token
    model = AutoModel.from_pretrained(cfg["model_name"], **kwargs)
    tokenizer = AutoTokenizer.from_pretrained(cfg["model_name"], **tok_kwargs)
    if not cfg.get("use_device_map"):
        model = model.to(DEVICE)
    model.eval()
    return model, tokenizer


def _extract_attn(outputs, num_layers: int, num_heads: int):
    """
    Extract attention matrices into a plain list[layer][head] → np.ndarray.
    Handles:
      - GQA models: HF returns shape [B, num_q_heads, S, S] after expansion
      - float16 tensors: cast to float32 before numpy
      - GPU tensors: .cpu() before .numpy()
    Returns None per layer/head slot on any failure.
    """
    result = []
    for l in range(num_layers):
        layer_heads = []
        for h in range(num_heads):
            try:
                att = outputs.attentions[l][0, h].float().detach().cpu().numpy()
                layer_heads.append(att)
            except Exception:
                layer_heads.append(None)
        result.append(layer_heads)
    return result


def load_done_keys(csv_path: str) -> set:
    done = set()
    if not os.path.exists(csv_path):
        return done
    try:
        df = pd.read_csv(csv_path)
        for _, row in df.iterrows():
            done.add((int(row["layer"]), int(row["head"]),
                      int(row["program_idx"]), int(row["sentence_idx"])))
    except Exception as e:
        print(f"[WARN] Could not read {csv_path}: {e}")
    return done


for model_key, cfg in MODEL_CONFIGS.items():
    out_csv = cfg["out_csv"]
    print(f"\n{'='*66}")
    print(f"[INFO] Model : {model_key}  ({cfg['model_name']})")
    print(f"[INFO] Output: {out_csv}")
    print(f"{'='*66}")

    try:
        model_obj, tokenizer_obj = _load_model(cfg)
    except Exception as e:
        print(f"[ERROR] Could not load {cfg['model_name']}: {e}")
        traceback.print_exc()
        continue

    num_layers = model_obj.config.num_hidden_layers
    num_heads  = model_obj.config.num_attention_heads
    # GQA info (informational only – HF already expands before returning attentions)
    num_kv = getattr(model_obj.config, "num_key_value_heads", num_heads)
    gqa    = num_kv < num_heads
    print(f"[INFO] Layers={num_layers} | Q-heads={num_heads} | KV-heads={num_kv}"
          f"{'  (GQA)' if gqa else ''} | fp16={cfg.get('use_fp16', False)}")

    # ── Step 1: cache attention tensors (N_sent forward passes) ────────────
    print("[INFO] Pre-computing model attentions …")
    attn_cache = []   # attn_cache[s_idx] = list[layer][head] → np.ndarray | None
    for s_idx, sentence in enumerate(SAMPLE_SENTENCES):
        try:
            tokens = tokenizer_obj(sentence, return_tensors="pt").to(DEVICE)
            with torch.no_grad():
                outputs = model_obj(**tokens, output_attentions=True)
            attn_cache.append(_extract_attn(outputs, num_layers, num_heads))
        except Exception as e:
            print(f"  [WARN] s_idx={s_idx} failed: {e}")
            attn_cache.append(None)
    n_ok = sum(1 for x in attn_cache if x is not None)
    print(f"[INFO] Attention cache ready ({n_ok}/{len(SAMPLE_SENTENCES)} sentences OK).")

    # ── Step 2: cache program outputs (pure numpy) ─────────────────────────
    print("[INFO] Pre-computing program outputs …")
    prog_cache = []   # prog_cache[p_idx][s_idx] → np.ndarray | None
    for prog in all_programs:
        prog_outputs = []
        for sentence in SAMPLE_SENTENCES:
            try:
                n_params = len(inspect.signature(prog).parameters)
                _, mat   = prog(sentence) if n_params == 1 else prog(sentence, tokenizer_obj)
                prog_outputs.append(np.array(mat, dtype=float))
            except Exception:
                prog_outputs.append(None)
        prog_cache.append(prog_outputs)
    print("[INFO] Program cache ready.")

    # ── Step 3: cache centroid matrices ───────────────────────────────────
    print("[INFO] Pre-computing centroid matrices …")
    centroid_cache: Dict[str, list] = {cat: [] for cat in CATEGORY_CENTROIDS}
    for cat, fn in CATEGORY_CENTROIDS.items():
        for sentence in SAMPLE_SENTENCES:
            try:
                _, cmat = fn(sentence, tokenizer_obj)
                centroid_cache[cat].append(np.array(cmat, dtype=float))
            except Exception:
                centroid_cache[cat].append(None)

    # ── Step 4: assign each (layer, head) to best category ────────────────
    print("[INFO] Assigning heads to categories …")
    cats = list(CATEGORY_CENTROIDS.keys())
    head_category: Dict[Tuple[int, int], str] = {}

    for layer in range(num_layers):
        for head in range(num_heads):
            cat_ious: Dict[str, list] = {c: [] for c in cats}
            for s_idx in range(len(SAMPLE_SENTENCES)):
                if attn_cache[s_idx] is None:
                    continue
                att = attn_cache[s_idx][layer][head]
                if att is None:
                    continue
                for cat in cats:
                    cmat = centroid_cache[cat][s_idx]
                    if cmat is None:
                        continue
                    n = min(att.shape[0], cmat.shape[0])
                    cat_ious[cat].append(float(np.mean(
                        [iou_score(att[i, :n], cmat[i, :n]) for i in range(n)]
                    )))
            means = {c: float(np.mean(v)) if v else 0.0 for c, v in cat_ious.items()}
            head_category[(layer, head)] = max(means, key=means.__getitem__)

    cat_dist   = Counter(head_category.values())
    total_full = num_layers * num_heads * len(all_programs) * len(SAMPLE_SENTENCES)
    total_cat  = sum(
        len(CATEGORY_PROG_INDICES.get(head_category[(l, h)], [])) * len(SAMPLE_SENTENCES)
        for l in range(num_layers) for h in range(num_heads)
    )
    print("[INFO] Head-to-category assignment:")
    for cat in cats:
        n_p = len(CATEGORY_PROG_INDICES.get(cat, []))
        print(f"  {cat:<30}: {cat_dist.get(cat, 0):>4} heads  | {n_p:>2} programs")
    pct_of_full = 100 * total_cat / total_full if total_full else 0
    print(f"[INFO] Scoring rows: {total_cat:,} / {total_full:,} full ({pct_of_full:.1f}%)")

    # ── Step 5: score and stream to CSV ──────────────────────────────────
    done_keys   = load_done_keys(out_csv)
    write_hdr   = not os.path.exists(out_csv) or os.path.getsize(out_csv) == 0
    errors = done_count = 0
    done_count  = len(done_keys)
    t0 = time.time()

    with open(out_csv, "a", newline="", encoding="utf-8") as fh:
        writer = csv.writer(fh)
        if write_hdr:
            writer.writerow(PART1_COLUMNS); fh.flush()

        for layer in range(num_layers):
            for head in range(num_heads):
                assigned_cat = head_category[(layer, head)]
                p_indices    = CATEGORY_PROG_INDICES.get(assigned_cat,
                                                          list(range(len(all_programs))))
                for p_idx in p_indices:
                    for s_idx in range(len(SAMPLE_SENTENCES)):
                        key = (layer, head, p_idx, s_idx)
                        if key in done_keys:
                            done_count += 1; continue

                        att  = attn_cache[s_idx][layer][head] if attn_cache[s_idx] else None
                        pred = prog_cache[p_idx][s_idx]

                        if att is None or pred is None:
                            score = float("nan")
                        else:
                            n = min(att.shape[0], pred.shape[0])
                            score = float(np.mean(
                                [iou_score(att[i, :n], pred[i, :n]) for i in range(n)]
                            ))

                        score_out = round(score, 3) if not np.isnan(score) else score
                        writer.writerow([layer, head, p_idx, s_idx, score_out])
                        fh.flush()
                        done_count += 1
                        if np.isnan(score): errors += 1

                        if done_count % 500 == 0:
                            elapsed = time.time() - t0
                            pct = done_count / total_cat * 100 if total_cat else 0
                            eta = elapsed / done_count * (total_cat - done_count) if done_count else 0
                            print(f"  [{model_key}] {done_count}/{total_cat} ({pct:.1f}%) | "
                                  f"elapsed {elapsed/60:.1f}m | ETA {eta/60:.1f}m | NaN={errors}")

    del model_obj, tokenizer_obj
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    print(f"[DONE] {model_key} complete. NaN={errors}")
    df_check = pd.read_csv(out_csv)
    print(f"[INFO] Rows in CSV: {len(df_check)}")

print("\n[DONE] Part 1 complete.")



[INFO] Model : bert  (bert-base-uncased)
[INFO] Output: ../data/iou_scores_bert.csv
[INFO] Layers=12 | Q-heads=12 | KV-heads=12 | fp16=False
[INFO] Pre-computing model attentions …
[INFO] Attention cache ready (5/5 sentences OK).
[INFO] Pre-computing program outputs …
[INFO] Program cache ready.
[INFO] Pre-computing centroid matrices …
[INFO] Assigning heads to categories …
[INFO] Head-to-category assignment:
  strict_anchors                :    6 heads  | 82 programs
  broadcasts_and_global         :   65 heads  | 10 programs
  structural_and_boundaries     :   61 heads  | 11 programs
  relative_flow                 :    8 heads  |  3 programs
  identity_and_local            :    1 heads  |  7 programs
  global_uniform                :    3 heads  | 17 programs
  sentence_start                :    0 heads  |  5 programs
  compound_element              :    0 heads  |  4 programs
[INFO] Scoring rows: 9,475 / 102,960 full (9.2%)
  [bert] 8500/9475 (89.7%) | elapsed 0.0m | ETA 0.0m | Na

In [10]:
# --- Part 1 verification ---
for model_key, cfg in MODEL_CONFIGS.items():
    csv_path = cfg["out_csv"]
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        print(
            f"[INFO] {model_key}: rows={len(df)} | NaN iou={df['iou_score'].isna().sum()} | "
            f"programs={df['program_idx'].nunique()} | mean_iou={df['iou_score'].mean():.4f}"
        )
    else:
        print(f"[WARN] {model_key}: CSV not found at {csv_path}")

[INFO] bert: rows=51120 | NaN iou=0 | programs=71 | mean_iou=0.1090
[INFO] gpt2: rows=51120 | NaN iou=288 | programs=71 | mean_iou=0.2613
[INFO] tinyllama: rows=237401 | NaN iou=0 | programs=68 | mean_iou=0.2909


## Part 2: Linear Interpolation Sweep (k = 1..20)

For each model and each attention head:
1. Rank programs by Part 1 mean IoU
2. Fit linear combinations using the top-k programs
3. Record IoU for each k

Output columns:
- model_key
- layer
- head
- k
- iou_score
- program_idxs_used

In [ ]:
# ── Part 2 helpers (GPU-aware) ──────────────────────────────────────────────

def linear_interp_iou(
    model: PreTrainedModel,
    tokenizer: PreTrainedTokenizerBase,
    layer: int,
    head: int,
    top_k_programs: List[Callable],
    sentences: List[str],
) -> float:
    """Fit a linear combination of top_k_programs; measure mean IoU."""
    all_ious = []
    for sentence in sentences:
        try:
            tokens = tokenizer(sentence, return_tensors="pt").to(DEVICE)
            with torch.no_grad():
                out = model(**tokens, output_attentions=True)
            att = out.attentions[layer][0, head].float().detach().cpu().numpy()

            X_rows = []
            for prog in top_k_programs:
                try:
                    n_params = len(inspect.signature(prog).parameters)
                    _, pred  = prog(sentence) if n_params == 1 else prog(sentence, tokenizer)
                    pred     = np.array(pred, dtype=float)
                    n        = min(att.shape[0], pred.shape[0])
                    X_rows.append(pred[:n, :n].flatten())
                except Exception:
                    return float("nan")

            seq_len  = att.shape[0]
            flat_len = seq_len * seq_len
            X_mat    = np.array([r[:flat_len] for r in X_rows]).T
            y_vec    = att.flatten()[:flat_len]
            X_mat    = np.nan_to_num(X_mat)
            y_vec    = np.nan_to_num(y_vec)

            reg    = LinearRegression(fit_intercept=True).fit(X_mat, y_vec)
            fitted = (reg.intercept_ + X_mat @ reg.coef_).reshape(seq_len, seq_len)
            row_s  = fitted.sum(axis=1, keepdims=True)
            fitted = fitted / np.where(row_s == 0, 1, row_s)
            fitted = np.clip(fitted, 1e-12, None)

            all_ious.append(float(np.mean([iou_score(att[i], fitted[i]) for i in range(seq_len)])))
        except Exception:
            all_ious.append(float("nan"))

    valid = [v for v in all_ious if not np.isnan(v)]
    return float(np.mean(valid)) if valid else float("nan")


K_VALUES      = list(range(1, 21))
PART2_COLUMNS = ["model_key", "layer", "head", "k", "iou_score", "program_idxs_used"]

print(f"[INFO] Part 2 K values: {K_VALUES}")
print(f"[INFO] Part 2 columns : {PART2_COLUMNS}")


In [ ]:
# --- PART 2: Linear interpolation sweep ---
# Strategy:
# 1) Use Part 1 CSV to rank programs per head by mean IoU
# 2) For each head and k in 1..20, fit linear combinations of top-k programs
# 3) Stream each row to CSV
# Resume-safe: skips (layer, head, k) that are already present.

INTERP_CONFIGS = {
    "bert":      f"{DATA_DIR}/interpolation_k_bert.csv",
    "gpt2":      f"{DATA_DIR}/interpolation_k_gpt2.csv",
    "tinyllama": f"{DATA_DIR}/interpolation_k_tinyllama.csv",
}


def load_done_interp(csv_path: str) -> set:
    done = set()
    if not os.path.exists(csv_path):
        return done
    try:
        df = pd.read_csv(csv_path)
        for _, row in df.iterrows():
            done.add((int(row["layer"]), int(row["head"]), int(row["k"])))
    except Exception as e:
        print(f"[WARN] Could not read {csv_path}: {e}")
    return done


for model_key in MODEL_CONFIGS:
    iou_csv = MODEL_CONFIGS[model_key]["out_csv"]
    out_csv = INTERP_CONFIGS[model_key]
    model_name = MODEL_CONFIGS[model_key]["model_name"]

    print(f"\n[INFO] {'=' * 64}")
    print(f"[INFO] Model: {model_key} ({model_name})")
    print(f"[INFO] Input Part 1 CSV: {iou_csv}")
    print(f"[INFO] Output Part 2 CSV: {out_csv}")
    print(f"[INFO] {'=' * 64}")

    if not os.path.exists(iou_csv):
        print(f"[WARN] Missing Part 1 CSV: {iou_csv}. Skipping.")
        continue

    iou_df = pd.read_csv(iou_csv)
    if iou_df.empty:
        print(f"[WARN] Part 1 CSV is empty for {model_key}. Skipping.")
        continue

    try:
        model_obj, tokenizer_obj = _load_model(cfg)
    except Exception as e:
        print(f"[ERROR] Could not load model {model_name}: {e}")
        traceback.print_exc()
        continue

    num_layers = model_obj.config.num_hidden_layers
    num_heads = model_obj.config.num_attention_heads

    # Mean IoU per (layer, head, program_idx), then rank programs per head
    head_rankings = {}
    grouped = (
        iou_df.groupby(["layer", "head", "program_idx"])["iou_score"]
        .mean()
        .reset_index()
    )
    for (layer, head), grp in grouped.groupby(["layer", "head"]):
        sorted_grp = grp.sort_values("iou_score", ascending=False)
        ordered_fns = []
        for p_idx in sorted_grp["program_idx"]:
            fn = all_programs[int(p_idx)] if int(p_idx) < len(all_programs) else None
            if fn is not None:
                ordered_fns.append(fn)
        head_rankings[(int(layer), int(head))] = ordered_fns

    total = num_layers * num_heads * len(K_VALUES)
    done_keys = load_done_interp(out_csv)
    print(f"[INFO] Resume keys loaded: {len(done_keys)} | Total target rows: {total}")

    write_header = not os.path.exists(out_csv) or os.path.getsize(out_csv) == 0
    errors = 0
    done_count = len(done_keys)
    t0 = time.time()

    with open(out_csv, "a", newline="", encoding="utf-8") as fh:
        writer = csv.writer(fh)
        if write_header:
            writer.writerow(PART2_COLUMNS)
            fh.flush()

        for layer in range(num_layers):
            for head in range(num_heads):
                ranked_fns = head_rankings.get((layer, head), all_programs)

                for k in K_VALUES:
                    key = (layer, head, k)
                    if key in done_keys:
                        done_count += 1
                        continue

                    top_k_fns = ranked_fns[:k]
                    if not top_k_fns:
                        top_k_fns = all_programs[:k]

                    iou = linear_interp_iou(
                        model_obj,
                        tokenizer_obj,
                        layer,
                        head,
                        top_k_fns,
                        SAMPLE_SENTENCES,
                    )

                    prog_idx_used = "|".join(str(all_programs.index(f)) for f in top_k_fns)
                    iou_out = round(iou, 6) if not np.isnan(iou) else iou
                    writer.writerow([model_key, layer, head, k, iou_out, prog_idx_used])
                    fh.flush()
                    done_count += 1

                    if np.isnan(iou):
                        errors += 1

                    if done_count % 200 == 0:
                        elapsed = time.time() - t0
                        pct = done_count / total * 100
                        eta = elapsed / done_count * (total - done_count) if done_count else 0
                        print(
                            f"[INFO] [{model_key}] {done_count}/{total} ({pct:.1f}%) | "
                            f"elapsed {elapsed/60:.1f}m | ETA {eta/60:.1f}m | NaN={errors}"
                        )

    del model_obj, tokenizer_obj
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print(f"[DONE] {model_key} interpolation complete. NaN scores: {errors}")
    df_check = pd.read_csv(out_csv)
    print(f"[INFO] Rows currently in CSV: {len(df_check)}")

print("\n[DONE] Part 2 complete.")

In [13]:
# --- Final verification summary ---
print("[INFO] === Part 1: IoU Scores ===")
for model_key, cfg in MODEL_CONFIGS.items():
    p = cfg["out_csv"]
    if os.path.exists(p):
        df = pd.read_csv(p)
        print(
            f"[INFO] {model_key}: rows={len(df)} | programs={df['program_idx'].nunique()} | "
            f"mean_iou={df['iou_score'].mean():.4f} | NaN={df['iou_score'].isna().sum()}"
        )
    else:
        print(f"[WARN] {model_key}: file not found at {p}")

print("\n[INFO] === Part 2: Interpolation Sweep ===")
for model_key, p in INTERP_CONFIGS.items():
    if os.path.exists(p):
        df = pd.read_csv(p)
        print(
            f"[INFO] {model_key}: rows={len(df)} | k_range={df['k'].min()}..{df['k'].max()} | "
            f"mean_iou={df['iou_score'].mean():.4f} | NaN={df['iou_score'].isna().sum()}"
        )
    else:
        print(f"[WARN] {model_key}: file not found at {p}")

print("\n[DONE] Verification summary complete.")

[INFO] === Part 1: IoU Scores ===
[INFO] bert: rows=51120 | programs=71 | mean_iou=0.1090 | NaN=0
[INFO] gpt2: rows=51120 | programs=71 | mean_iou=0.2613 | NaN=288
[INFO] tinyllama: rows=237401 | programs=68 | mean_iou=0.2909 | NaN=0

[INFO] === Part 2: Interpolation Sweep ===
[INFO] bert: rows=2880 | k_range=1..20 | mean_iou=0.4760 | NaN=0
[INFO] gpt2: rows=2880 | k_range=1..20 | mean_iou=0.5893 | NaN=8
[INFO] tinyllama: rows=13088 | k_range=1..20 | mean_iou=0.6961 | NaN=0

[DONE] Verification summary complete.
